<a href="https://colab.research.google.com/github/ParusSlava/DTA_2026/blob/main/ML/ML_feature_engineering_categorical_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature engineering. Categorical features: One-Hot, Ordinal

## Налаштування та дані

- квартири (регресія) — з числовими та категорійними ознаками й датою

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# ---------- Датасет 1: КВАРТИРИ (регресія) ----------
n = 1500
cities = np.random.choice(["Київ", "Львів", "Харків", "Одеса"], n, p=[.4, .2, .2, .2])
city_premium = pd.Series({"Київ": 60, "Львів": 25, "Харків": 10, "Одеса": 20})

condition = np.random.choice(["аварійний", "житловий", "хороший", "євроремонт"],
                             n, p=[.1, .4, .35, .15])
cond_bonus = pd.Series({"аварійний": -20, "житловий": 0, "хороший": 15, "євроремонт": 40})

area    = np.random.normal(60, 20, n).clip(20, 140)
rooms   = np.clip(np.round(area / 25 + np.random.normal(0, .6, n)), 1, 5).astype(int)
floor   = np.random.randint(1, 25, n)
dist_km = np.random.exponential(5, n).clip(.3, 25)
listing_date = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 540, n), unit="D")

price = (40 + area*1.8 + rooms*5 + floor*.4 - dist_km*3
         + city_premium[cities].values + cond_bonus[condition].values
         + np.random.normal(0, 12, n)).clip(20, None)

apt = pd.DataFrame({
    "area": area.round(1), "rooms": rooms, "floor": floor,
    "dist_km": dist_km.round(1), "city": cities, "condition": condition,
    "listing_date": listing_date, "price": price.round(1),
})


print("Квартири:", apt.shape)
apt.head()

Квартири: (1500, 8)


,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


## Feature Engineering — створення нових ознак

In [3]:
df = apt.copy()
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


In [8]:
# Співвідношення
df['area_per_room'] = (df.area / df.rooms).round(1)
df[['area', 'rooms', 'area_per_room']].head()

,area,rooms,area_per_room
0,81.2,3,27.1
1,72.3,3,24.1
2,73.7,4,18.4
3,32.7,1,32.7
4,84.2,3,28.1


In [14]:
# Взаємодія: пошук великої квартири, яка знаходиться далеко від центру
df['area_x_dist'] = (df.area * df.dist_km).round(1)
df[['area', 'dist_km', 'area_x_dist']].head()
df[['area', 'dist_km', 'area_x_dist']].describe()

,area,dist_km,area_x_dist
count,1500.000000,1500.000000,1500.000000
mean,60.012067,4.689333,281.348067
std,19.851969,4.516440,298.656168
min,20.000000,0.300000,6.000000
25%,45.600000,1.475000,74.150000
50%,59.750000,3.200000,182.050000
75%,73.400000,6.400000,377.150000
max,138.500000,25.000000,1880.000000


In [16]:
# Бінарна ознака: True(1) & False(0)
df['is_cantral'] = (df.dist_km < 3).astype(int)
df[['dist_km', 'is_cantral']].head()

,dist_km,is_cantral
0,4.4,0
1,4.0,0
2,1.2,1
3,2.8,1
4,5.3,0


In [20]:
# Бін (Бінінг) - поділ на групи
df['floor_group'] = pd.cut(
    df.floor,
    bins=[0, 2, 9, 100],
    labels = ['low', 'middle', 'high']
)

df[['floor', 'floor_group']].head()

,floor,floor_group
0,20,high
1,10,high
2,23,high
3,9,middle
4,2,low
